In [1]:
%pip install langchain langchain-community langchain-experimental

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

DATASET_DIR = Path(r"C:\Users\BIT\Downloads\datasets\datasets")
EQUIPMENT_XLSX = DATASET_DIR / "equipment_register" / "equipment_register.xlsx"
MAINTENANCE_XLSX = DATASET_DIR / "maintenance_reports" / "maintenance_reports.xlsx"
INSPECTION_XLSX = DATASET_DIR / "inspection_reports" / "inspection_reports.xlsx"
INCIDENT_XLSX = DATASET_DIR / "incident_reports" / "incident_reports.xlsx"

CHROMA_DIR = Path("./chroma_db")            # reuse the store built in notebook 1
CHROMA_COLLECTION = "plant_documents"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

GEMINI_MODEL = "gemini-3.5-flash"
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY as an environment variable before running the LLM cells below."

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, google_api_key=GEMINI_API_KEY, temperature=0)


In [3]:
import pandas as pd
import duckdb

df_equipment = pd.read_excel(EQUIPMENT_XLSX)
df_maintenance = pd.read_excel(MAINTENANCE_XLSX)
df_inspection = pd.read_excel(INSPECTION_XLSX)
df_incident = pd.read_excel(INCIDENT_XLSX)

for name, df in [("equipment", df_equipment), ("maintenance", df_maintenance),
                  ("inspection", df_inspection), ("incident", df_incident)]:
    print(name, df.shape)

DUCKDB_PATH = "plant_data.duckdb"
con = duckdb.connect(DUCKDB_PATH)
con.execute("CREATE OR REPLACE TABLE equipment AS SELECT * FROM df_equipment")
con.execute("CREATE OR REPLACE TABLE maintenance AS SELECT * FROM df_maintenance")
con.execute("CREATE OR REPLACE TABLE inspection AS SELECT * FROM df_inspection")
con.execute("CREATE OR REPLACE TABLE incident AS SELECT * FROM df_incident")
con.close()
print("DuckDB file written to", DUCKDB_PATH)


equipment (22, 12)
maintenance (87, 15)
inspection (75, 14)
incident (25, 17)
DuckDB file written to plant_data.duckdb


In [5]:
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
import re


# Connect DuckDB
sql_db = SQLDatabase.from_uri(f"duckdb:///{DUCKDB_PATH}")

# SQL generation chain
write_sql_query = create_sql_query_chain(llm, sql_db)

# SQL execution tool
run_sql_query = QuerySQLDataBaseTool(db=sql_db)


def clean_sql(sql: str) -> str:
    """Clean SQL generated by the LLM."""

    sql = str(sql).strip()

    # Remove SQLQuery: prefix if present
    sql = re.sub(r"^\s*SQLQuery\s*:\s*", "", sql, flags=re.IGNORECASE)

    # Remove markdown code fences
    if sql.startswith(""):
        lines = sql.splitlines()

        # Remove first line: sql or 
        if lines and lines[0].strip().lower() in ("sql", ""):
            lines = lines[1:]

        # Remove last 
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        sql = "\n".join(lines).strip()

    # Remove stray "sql" at the beginning
    if sql.lower().startswith("sql"):
        sql = sql[3:].strip()

    return sql


def structured_search(question: str):

    # Generate SQL
    sql = write_sql_query.invoke({
        "question": question
    })

    # Clean generated SQL
    sql = clean_sql(sql)

    # Only allow SELECT queries
    if not sql.upper().startswith("SELECT"):
        return {
            "sql": sql,
            "result": None,
            "error": "Refused: generated query was not a SELECT"
        }

    # Execute SQL
    try:
        result = run_sql_query.invoke(sql)

        return {
            "sql": sql,
            "result": result,
            "error": None
        }

    except Exception as e:

        return {
            "sql": sql,
            "result": None,
            "error": str(e)
        }


# Quick test
out = structured_search(
    "How many maintenance records are there for pumps?"
)

print("SQL:", out["sql"])
print("Result:", out["result"])
print("Error:", out["error"])

c:\Users\BIT\Management Copilot\rag_env\lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

In [4]:
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

sql_db = SQLDatabase.from_uri(f"duckdb:///{DUCKDB_PATH}")

write_sql_query = create_sql_query_chain(llm, sql_db)
run_sql_query = QuerySQLDataBaseTool(db=sql_db)


def structured_search(question: str):
    sql = write_sql_query.invoke({"question": question})

    sql = sql.split("SQLQuery:")[-1].strip()

    if sql.startswith(""):
        lines = sql.splitlines()

        if lines and lines[0].strip().lower() in ("sql", ""):
            lines = lines[1:]

        if lines and lines[-1].strip() == "":
            lines = lines[:-1]

        sql = "\n".join(lines).strip()

    if sql.lower().startswith("sql"):
        sql = sql[3:].strip()

    if not sql.upper().startswith("SELECT"):
        return {
            "sql": sql,
            "result": None,
            "error": "Refused: generated query was not a SELECT"
        }

    try:
        result = run_sql_query.invoke(sql)

        return {
            "sql": sql,
            "result": result,
            "error": None
        }

    except Exception as e:
        return {
            "sql": sql,
            "result": None,
            "error": str(e)
        }

# quick test
out = structured_search("How many maintenance records are there for pumps?")
print("SQL:", out["sql"])
print("Result:", out["result"])


C:\Users\BIT\AppData\Local\Temp\ipykernel_10496\3151264163.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase
c:\Users\BIT\Management Copilot\rag_env\lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(
C:\Users\BIT\AppData\Local\Temp\ipykernel_10496\3151264163.py:8: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  run_sql_query = QuerySQLDataBaseT

ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self signed certificate in certificate chain (_ssl.c:1007)

In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [25]:
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

driver.verify_connectivity()
print("✅ Neo4j connected!")

✅ Neo4j connected!


In [6]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


def clear_graph(tx):
    tx.run("MATCH (n) DETACH DELETE n")

def load_equipment(tx, row):
    tx.run("""
        MERGE (e:Equipment {equipment_id: $equipment_id})
        SET e.equipment_code = $equipment_code, e.name = $name, e.type = $type,
            e.manufacturer = $manufacturer, e.model = $model, e.status = $status,
            e.criticality = $criticality, e.linked_manual = $linked_manual
        MERGE (man:Manual {name: $linked_manual})
        MERGE (e)-[:HAS_MANUAL]->(man)
    """,
        equipment_id=str(row.get("equipment_id")), equipment_code=str(row.get("equipment_code")),
        name=str(row.get("equipment_name")), type=str(row.get("equipment_type")),
        manufacturer=str(row.get("manufacturer")), model=str(row.get("model")),
        status=str(row.get("status")), criticality=str(row.get("Criticality")),
        linked_manual=str(row.get("linked_manual")))

def load_maintenance(tx, row):
    tx.run("""
        MATCH (e:Equipment {equipment_id: $equipment_id})
        MERGE (m:MaintenanceRecord {report_id: $report_id})
        SET m.maintenance_type = $maintenance_type, m.date = $date,
            m.issue_reported = $issue_reported, m.action_taken = $action_taken,
            m.downtime_hours = $downtime_hours
        MERGE (e)-[:UNDERWENT_MAINTENANCE]->(m)
    """,
        equipment_id=str(row.get("equipment_id")), report_id=str(row.get("report_id")),
        maintenance_type=str(row.get("maintenance_type")), date=str(row.get("date")),
        issue_reported=str(row.get("issue_reported")), action_taken=str(row.get("action_taken")),
        downtime_hours=str(row.get("downtime_hours")))

def load_inspection(tx, row):
    tx.run("""
        MATCH (e:Equipment {equipment_id: $equipment_id})
        MERGE (i:InspectionRecord {inspection_id: $inspection_id})
        SET i.inspection_date = $inspection_date, i.condition = $condition,
            i.recommendation = $recommendation
        MERGE (e)-[:WAS_INSPECTED]->(i)
    """,
        equipment_id=str(row.get("equipment_id")), inspection_id=str(row.get("inspection_id")),
        inspection_date=str(row.get("inspection_date")), condition=str(row.get("condition")),
        recommendation=str(row.get("recommendation")))

def load_incident(tx, row):
    tx.run("""
        MATCH (e:Equipment {equipment_id: $equipment_id})
        MERGE (x:IncidentRecord {incident_id: $incident_id})
        SET x.incident_date = $incident_date, x.severity = $severity,
            x.root_cause = $root_cause, x.corrective_action = $corrective_action,
            x.downtime_hours = $downtime_hours
        MERGE (e)-[:HAD_INCIDENT]->(x)
    """,
        equipment_id=str(row.get("equipment_id")), incident_id=str(row.get("incident_id")),
        incident_date=str(row.get("incident_date")), severity=str(row.get("severity")),
        root_cause=str(row.get("root_cause")), corrective_action=str(row.get("corrective_action")),
        downtime_hours=str(row.get("downtime_hours")))


with driver.session() as session:
    session.execute_write(clear_graph)
    for _, row in df_equipment.iterrows():
        session.execute_write(load_equipment, row)
    for _, row in df_maintenance.iterrows():
        session.execute_write(load_maintenance, row)
    for _, row in df_inspection.iterrows():
        session.execute_write(load_inspection, row)
    for _, row in df_incident.iterrows():
        session.execute_write(load_incident, row)

print("Graph built:")
print(" ", len(df_equipment), "equipment nodes")
print(" ", len(df_maintenance), "maintenance records")
print(" ", len(df_inspection), "inspection records")
print(" ", len(df_incident), "incident records")


Graph built:
  22 equipment nodes
  87 maintenance records
  75 inspection records
  25 incident records


In [7]:
from langchain_community.graphs import Neo4jGraph
from langchain_classic.chains import GraphCypherQAChain

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USER, password=NEO4J_PASSWORD)
graph.refresh_schema()
print(graph.schema)

graph_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
      return_direct=True  # we trust this LLM call in this local/dev context
)

# quick test
result = graph_chain.invoke({"query": "Which equipment had an incident and also needed maintenance?"})
print(result["result"])

C:\Users\BIT\AppData\Local\Temp\ipykernel_20108\65782115.py:4: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USER, password=NEO4J_PASSWORD)


Node properties:
Equipment {equipment_id: STRING, equipment_code: STRING, name: STRING, type: STRING, manufacturer: STRING, model: STRING, status: STRING, criticality: STRING, linked_manual: STRING}
Manual {name: STRING}
MaintenanceRecord {report_id: STRING, maintenance_type: STRING, date: STRING, issue_reported: STRING, action_taken: STRING, downtime_hours: STRING}
InspectionRecord {inspection_id: STRING, inspection_date: STRING, condition: STRING, recommendation: STRING}
IncidentRecord {downtime_hours: STRING, incident_id: STRING, incident_date: STRING, severity: STRING, root_cause: STRING, corrective_action: STRING}
Relationship properties:

The relationships:
(:Equipment)-[:HAS_MANUAL]->(:Manual)
(:Equipment)-[:UNDERWENT_MAINTENANCE]->(:MaintenanceRecord)
(:Equipment)-[:WAS_INSPECTED]->(:InspectionRecord)
(:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord)


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (e:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord), (e)-[

In [8]:
result = graph.query("""
MATCH (e:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord)
RETURN DISTINCT e.equipment_code AS equipment_code
""")

print(result)

[{'equipment_code': 'P-101'}, {'equipment_code': 'P-102'}, {'equipment_code': 'P-103'}, {'equipment_code': 'M-101'}, {'equipment_code': 'M-102'}, {'equipment_code': 'M-103'}, {'equipment_code': 'C-101'}, {'equipment_code': 'C-102'}, {'equipment_code': 'B-101'}, {'equipment_code': 'HX-101'}, {'equipment_code': 'HX-102'}, {'equipment_code': 'FM-101'}, {'equipment_code': 'FM-102'}, {'equipment_code': 'CV-101'}, {'equipment_code': 'CV-102'}, {'equipment_code': 'CN-101'}, {'equipment_code': 'VFD-101'}, {'equipment_code': 'VFD-102'}, {'equipment_code': 'GB-101'}, {'equipment_code': 'GB-102'}]


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = Chroma(
    collection_name=CHROMA_COLLECTION,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("Vector store loaded, collection size:", vectorstore._collection.count())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store loaded, collection size: 10624


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

ROUTE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Classify a plant-maintenance question into one or more routes:
- vector: needs free-text knowledge from manuals/SOPs (procedures, specs, safety steps)
- structured: needs counting/filtering/aggregating over tabular logs
- graph: needs multi-hop relationships between equipment, maintenance, inspections, incidents

Return ONLY a comma-separated list using these three words, nothing else."""),
    ("human", "{question}"),
])
route_chain = ROUTE_PROMPT | llm | StrOutputParser()


def classify_routes(question: str):
    raw = route_chain.invoke({"question": question})
    routes = [r.strip().lower() for r in raw.split(",") if r.strip().lower() in ("vector", "structured", "graph")]
    return routes or ["vector"]


ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a plant maintenance and safety assistant.
Answer using ONLY the evidence provided. Cite the source file/page for manual/SOP facts,
and the record id for structured/graph facts. If evidence is missing, say so plainly."""),
    ("human", "Question: {question}\n\nEvidence:\n{evidence}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def gather_evidence(question, routes):
    evidence_parts = []
    detail = {}

    if "vector" in routes:
        docs = retriever.invoke(question)
        detail["vector"] = docs
        evidence_parts.append("DOCUMENT EXCERPTS:")
        for d in docs:
            evidence_parts.append(f"- [{d.metadata.get('source_file')}, p.{d.metadata.get('page')}] {d.page_content[:400]}")

    if "structured" in routes:
        s = structured_search(question)
        detail["structured"] = s
        evidence_parts.append(f"\nSTRUCTURED QUERY (SQL): {s['sql']}")
        evidence_parts.append(f"Result: {s['result'] if not s['error'] else s['error']}")

    if "graph" in routes:
        g = graph_chain.invoke({"query": question})
        detail["graph"] = g
        evidence_parts.append(f"\nGRAPH ANSWER: {g['result']}")

    return "\n".join(evidence_parts), detail


def answer_question(question: str):
    routes = classify_routes(question)
    evidence_text, detail = gather_evidence(question, routes)
    final_answer = answer_chain.invoke({"question": question, "evidence": evidence_text})
    return {"routes": routes, "answer": final_answer, "detail": detail}


In [11]:
r = answer_question("How many maintenance events happened on the centrifugal pump this year?")
print("Routes used:", r["routes"])
print(r["answer"])



Routes used: ['structured']
Based on the provided structured query result, there have been **0** maintenance events on the centrifugal pump this year (Result: `[(0,)]`).


In [12]:
r = answer_question("Which equipment had an incident and also needed bearing-related maintenance?")
print("Routes used:", r["routes"])
print(r["answer"])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (e:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord)
MATCH (e)-[:UNDERWENT_MAINTENANCE]->(m:MaintenanceRecord)
WHERE toLower(m.issue_reported) CONTAINS 'bearing' OR toLower(m.action_taken) CONTAINS 'bearing'
RETURN DISTINCT e


> Finished chain.
Routes used: ['graph']
Based on the provided evidence, there is no information regarding incidents or bearing-related maintenance for the listed equipment (Cooling Water Pump `EQ001` / `P-101` and Cooling Water Pump Motor `EQ004` / `M-101`). 

Therefore, the evidence required to answer which equipment had an incident and needed bearing-related maintenance is missing.


In [13]:
result = graph.query("""
MATCH (e:Equipment)-[:UNDERWENT_MAINTENANCE]->(m:MaintenanceRecord)
RETURN e.equipment_code AS equipment_code,
       m.issue_reported AS issue,
       m.action_taken AS action
""")

print(result)

[{'equipment_code': 'P-101', 'issue': 'High vibration', 'action': 'Lubricated bearing and adjusted suction pressure'}, {'equipment_code': 'P-101', 'issue': 'Bearing wear detected', 'action': 'Replaced drive-end bearing'}, {'equipment_code': 'P-101', 'issue': 'Seal leakage observed', 'action': 'Tightened seal gland, ordered replacement seal'}, {'equipment_code': 'P-101', 'issue': 'Pump restored after overhaul', 'action': 'Replaced mechanical seal and conducted full alignment'}, {'equipment_code': 'P-102', 'issue': 'Cavitation noise detected', 'action': 'Adjusted suction valve'}, {'equipment_code': 'P-102', 'issue': 'Minor seal weeping', 'action': 'Tightened seal bolts'}, {'equipment_code': 'P-102', 'issue': 'Routine vibration route', 'action': 'Collected baseline vibration data'}, {'equipment_code': 'P-102', 'issue': 'Routine vibration route', 'action': 'Collected baseline vibration data'}, {'equipment_code': 'P-103', 'issue': 'Minor seal weeping', 'action': 'Tightened seal bolts'}, {'e

In [14]:
result = graph.query("""
MATCH (e:Equipment)-[:HAD_INCIDENT]->(i:IncidentRecord)
MATCH (e)-[:UNDERWENT_MAINTENANCE]->(m:MaintenanceRecord)
RETURN e.equipment_code AS equipment_code,
       i.issue_reported AS incident,
       m.issue_reported AS maintenance_issue,
       m.action_taken AS action
""")

print(result)

[{'equipment_code': 'P-101', 'incident': None, 'maintenance_issue': 'High vibration', 'action': 'Lubricated bearing and adjusted suction pressure'}, {'equipment_code': 'P-101', 'incident': None, 'maintenance_issue': 'Bearing wear detected', 'action': 'Replaced drive-end bearing'}, {'equipment_code': 'P-101', 'incident': None, 'maintenance_issue': 'Seal leakage observed', 'action': 'Tightened seal gland, ordered replacement seal'}, {'equipment_code': 'P-101', 'incident': None, 'maintenance_issue': 'Pump restored after overhaul', 'action': 'Replaced mechanical seal and conducted full alignment'}, {'equipment_code': 'P-101', 'incident': None, 'maintenance_issue': 'High vibration', 'action': 'Lubricated bearing and adjusted suction pressure'}, {'equipment_code': 'P-101', 'incident': None, 'maintenance_issue': 'Bearing wear detected', 'action': 'Replaced drive-end bearing'}, {'equipment_code': 'P-101', 'incident': None, 'maintenance_issue': 'Seal leakage observed', 'action': 'Tightened seal

In [15]:
result = graph_chain.invoke({
    "query": "Which equipment had an incident and also underwent maintenance?"
})

print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (e:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord), (e)-[:UNDERWENT_MAINTENANCE]->(:MaintenanceRecord)
RETURN e


> Finished chain.
[{'e': {'equipment_code': 'P-101', 'linked_manual': 'centrifugal_pump_oem_manual.pdf', 'name': 'Cooling Water Pump', 'criticality': 'High', 'model': 'CP-500', 'type': 'Pump', 'equipment_id': 'EQ001', 'manufacturer': 'APV', 'status': 'Running'}}, {'e': {'equipment_code': 'P-101', 'linked_manual': 'centrifugal_pump_oem_manual.pdf', 'name': 'Cooling Water Pump', 'criticality': 'High', 'model': 'CP-500', 'type': 'Pump', 'equipment_id': 'EQ001', 'manufacturer': 'APV', 'status': 'Running'}}, {'e': {'equipment_code': 'P-101', 'linked_manual': 'centrifugal_pump_oem_manual.pdf', 'name': 'Cooling Water Pump', 'criticality': 'High', 'model': 'CP-500', 'type': 'Pump', 'equipment_id': 'EQ001', 'manufacturer': 'APV', 'status': 'Running'}}, {'e': {'equipment_code': 'P-101', 'linked_manual': 'centri

In [16]:
result = graph_chain.invoke({
    "query": "Which equipment had an incident and underwent maintenance? Answer only with the equipment code."
})

print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord), (e)-[:UNDERWENT_MAINTENANCE]->(:MaintenanceRecord)
RETURN e.equipment_code

> Finished chain.
[{'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-102'}, {'e.equipment_code': 'P-102'}]


In [17]:
result = graph_chain.invoke({
    "query": "Which equipment had an incident and also underwent maintenance? Return only unique equipment codes."
})

print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (e:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord)
MATCH (e)-[:UNDERWENT_MAINTENANCE]->(:MaintenanceRecord)
RETURN DISTINCT e.equipment_code


> Finished chain.
[{'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-102'}, {'e.equipment_code': 'P-103'}, {'e.equipment_code': 'M-101'}, {'e.equipment_code': 'M-102'}, {'e.equipment_code': 'M-103'}, {'e.equipment_code': 'C-101'}, {'e.equipment_code': 'C-102'}, {'e.equipment_code': 'B-101'}, {'e.equipment_code': 'HX-101'}]


In [18]:
result = graph_chain.invoke({
    "query": "Which equipment had an incident and also underwent maintenance? Return only unique equipment codes."
})

print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (e:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord)
MATCH (e)-[:UNDERWENT_MAINTENANCE]->(:MaintenanceRecord)
RETURN DISTINCT e.equipment_code


> Finished chain.
[{'e.equipment_code': 'P-101'}, {'e.equipment_code': 'P-102'}, {'e.equipment_code': 'P-103'}, {'e.equipment_code': 'M-101'}, {'e.equipment_code': 'M-102'}, {'e.equipment_code': 'M-103'}, {'e.equipment_code': 'C-101'}, {'e.equipment_code': 'C-102'}, {'e.equipment_code': 'B-101'}, {'e.equipment_code': 'HX-101'}]


In [20]:
r = answer_question("Which equipment had an incident and also needed bearing-related maintenance?")
print("Routes used:", r["routes"])
print(r["answer"])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (e:Equipment)-[:HAD_INCIDENT]->(:IncidentRecord)
MATCH (e)-[:UNDERWENT_MAINTENANCE]->(m:MaintenanceRecord)
WHERE toLower(m.issue_reported) CONTAINS 'bearing' OR toLower(m.action_taken) CONTAINS 'bearing'
RETURN DISTINCT e


> Finished chain.
Routes used: ['graph']
Based on the provided evidence, there is no information regarding incidents or bearing-related maintenance for any of the listed equipment (EQ001 or EQ004). The necessary evidence to answer this question is missing.


In [26]:
with driver.session() as session:
    result = session.run("""
        MATCH (e:Equipment)-[:UNDERWENT_MAINTENANCE]->(m:MaintenanceRecord)
        RETURN e.name AS equipment, m.issue_reported AS issue, m.action_taken AS action
        LIMIT 10
    """)

    for record in result:
        print(record)

<Record equipment='Cooling Water Pump' issue='High vibration' action='Lubricated bearing and adjusted suction pressure'>
<Record equipment='Cooling Water Pump' issue='Bearing wear detected' action='Replaced drive-end bearing'>
<Record equipment='Cooling Water Pump' issue='Seal leakage observed' action='Tightened seal gland, ordered replacement seal'>
<Record equipment='Cooling Water Pump' issue='Pump restored after overhaul' action='Replaced mechanical seal and conducted full alignment'>
<Record equipment='Feed Water Pump' issue='Cavitation noise detected' action='Adjusted suction valve'>
<Record equipment='Feed Water Pump' issue='Minor seal weeping' action='Tightened seal bolts'>
<Record equipment='Feed Water Pump' issue='Routine vibration route' action='Collected baseline vibration data'>
<Record equipment='Feed Water Pump' issue='Routine vibration route' action='Collected baseline vibration data'>
<Record equipment='Boiler Feed Pump' issue='Minor seal weeping' action='Tightened seal

In [19]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

question_box = widgets.Text(
    placeholder="Ask about your plant... e.g. What is the boiler startup procedure?",
    layout=widgets.Layout(width="80%"),
)
ask_button = widgets.Button(description="Ask", button_style="primary")
output_area = widgets.Output()

def on_ask_clicked(b):
    with output_area:
        clear_output()
        q = question_box.value.strip()
        if not q:
            print("Type a question first.")
            return
        display(Markdown(f"**Q:** {q}"))
        result = answer_question(q)
        display(Markdown(f"**Routes used:** {', '.join(result['routes'])}"))
        display(Markdown(f"**A:** {result['answer']}"))

ask_button.on_click(on_ask_clicked)

display(widgets.HBox([question_box, ask_button]))
display(output_area)


Output()